In [ ]:
%pip -q install "transformers==5.22.1"

import csv
import hashlib
import json
import shutil
import tarfile
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/digikala-project-3")
reports = root / "reports"
original_dir = root / "artifacts" / "parsbert_supervised"
checkpoint_path = root / "artifacts" / "parsbert_supervised_1200k" / "best.pt"
output_dir = root / "artifacts" / "parsbert_semi_supervised_1200k"
parts_dir = output_dir / "scoring_parts"
parts_dir.mkdir(parents=True, exist_ok=True)

splits = Path("/content/digikala_splits")
pool = splits / "train_pseudo_unlabeled.csv"
manifest_path = splits / "split_manifest.json"

if not pool.is_file() or not manifest_path.is_file():
    bundle = Path("/content/digikala_inputs.tar.gz")
    shutil.copyfile(root / bundle.name, bundle)
    with tarfile.open(bundle, "r:gz") as archive:
        archive.extractall("/content", filter="data")

for name in [
    "digikala_transformer.py", "digikala_training.py",
    "digikala_evaluation.py", "normalize_persian.py",
]:
    shutil.copyfile(original_dir / name, Path("/content") / name)

from digikala_transformer import REC_LABELS, DigikalaTransformer
from normalize_persian import normalize_text

if not torch.cuda.is_available():
    raise RuntimeError("Select T4 GPU in Runtime > Change runtime type.")

training = json.loads(
    (reports / "parsbert_supervised_1200k.json").read_text(encoding="utf-8")
)
settings = training["settings"]
assert settings["train_rows"] == 1_200_000

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert checkpoint["epoch"] == training["parent_epoch"] + training["additional_epochs"]

THRESHOLD = 0.90
BATCH_SIZE = 256
CHUNK_ROWS = 25_000

def sha256_file(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

scoring_config = {
    "checkpoint_epoch": checkpoint["epoch"],
    "checkpoint_sha256": sha256_file(checkpoint_path),
    "pool_sha256": sha256_file(pool),
    "threshold": THRESHOLD,
    "batch_size": BATCH_SIZE,
    "chunk_rows": CHUNK_ROWS,
    "max_length": settings["max_length"],
}
config_path = parts_dir / "scoring_config.json"
if config_path.is_file():
    assert json.loads(config_path.read_text(encoding="utf-8")) == scoring_config
else:
    config_path.write_text(json.dumps(scoring_config, indent=2), encoding="utf-8")

tokenizer = AutoTokenizer.from_pretrained(
    str(original_dir / "tokenizer"), local_files_only=True,
)
encoder = AutoModel.from_pretrained(
    settings["model"], revision=settings["revision"], add_pooling_layer=False,
    use_safetensors=False, weights_only=True, trust_remote_code=False,
)
model = DigikalaTransformer(encoder).to("cuda")
assert set(checkpoint["parameters"]) == {
    name for name, parameter in model.named_parameters() if parameter.requires_grad
}
model.load_state_dict(checkpoint["parameters"], strict=False)
model.eval()

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
expected_rows = manifest["exported_file_rows"][pool.name]
label_names = {code: label for label, code in REC_LABELS.items()}
chunks = pd.read_csv(pool, chunksize=CHUNK_ROWS, dtype=str, keep_default_na=False)
part_paths = []
total_rows = 0

with torch.inference_mode():
    for index, chunk in enumerate(tqdm(
        chunks, total=(expected_rows + CHUNK_ROWS - 1) // CHUNK_ROWS,
        desc="Pseudo-labeling",
    )):
        assert list(chunk.columns) == ["id", "body"]
        total_rows += len(chunk)
        part = parts_dir / f"part_{index:03d}.csv"
        part_paths.append(part)
        if part.is_file():
            continue

        temporary = part.with_suffix(".tmp")
        with temporary.open("w", encoding="utf-8", newline="") as file:
            writer = csv.writer(file)
            for start in range(0, len(chunk), BATCH_SIZE):
                batch = chunk.iloc[start:start + BATCH_SIZE]
                inputs = tokenizer(
                    batch["body"].map(normalize_text).tolist(),
                    truncation=True, max_length=settings["max_length"],
                    padding=True, pad_to_multiple_of=8, return_tensors="pt",
                ).to("cuda")

                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits, _ = model(**inputs)

                confidence, codes = logits.float().softmax(dim=1).max(dim=1)
                for row, probability, code in zip(
                    batch.itertuples(index=False),
                    confidence.cpu().tolist(),
                    codes.cpu().tolist(),
                ):
                    if probability >= THRESHOLD:
                        writer.writerow([row.id, row.body, label_names[code], probability])
        temporary.replace(part)

assert total_rows == expected_rows
accepted_counts = Counter()
destination = output_dir / "pseudo_labeled.csv"
temporary = destination.with_suffix(".tmp")

with temporary.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["id", "body", "recommendation_status", "confidence"])
    for part in part_paths:
        with part.open(encoding="utf-8", newline="") as source:
            for row in csv.reader(source):
                writer.writerow(row)
                accepted_counts[row[2]] += 1

temporary.replace(destination)
accepted = sum(accepted_counts.values())
report = {
    **scoring_config,
    "source_pool": pool.name,
    "scored_rows": total_rows,
    "accepted_rows": accepted,
    "accepted_pct": 100 * accepted / total_rows,
    "accepted_by_class": {label: accepted_counts[label] for label in REC_LABELS},
}
(reports / "pseudo_labeling_1200k.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8",
)

print(f"Scored: {total_rows:,} | Accepted: {accepted:,} ({accepted / total_rows:.2%})")
for label in REC_LABELS:
    print(f"{label:16s} {accepted_counts[label]:,}")
print("Saved: pseudo_labeled.csv and pseudo_labeling_1200k.json")

Mounted at /content/drive


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

Pseudo-labeling:   0%|          | 0/30 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Scored: 736,540 | Accepted: 279,363 (37.93%)
recommended      256,238
not_recommended  22,594
no_idea          531
Saved: pseudo_labeled.csv and pseudo_labeling_1200k.json


In [ ]:
%pip -q install duckdb

import json
import math
from copy import deepcopy
from time import perf_counter

import duckdb
import pandas as pd
import torch

from digikala_training import make_loader, run_epoch

pseudo_report = json.loads(
    (reports / "pseudo_labeling_1200k.json").read_text(encoding="utf-8")
)
assert pseudo_report["threshold"] == 0.90
assert pseudo_report["checkpoint_sha256"] == sha256_file(checkpoint_path)

pseudo = pd.read_csv(
    output_dir / "pseudo_labeled.csv",
    dtype={"id": str, "body": str, "recommendation_status": str, "confidence": float},
    keep_default_na=False,
)
assert list(pseudo.columns) == ["id", "body", "recommendation_status", "confidence"]
assert len(pseudo) == pseudo_report["accepted_rows"]
assert pseudo["id"].is_unique
assert pseudo["recommendation_status"].isin(REC_LABELS).all()
assert pseudo["confidence"].between(0.90, 1.0).all()

train_rows = settings["train_rows"]
validation_rows = settings["validation_rows_for_checkpoint_selection"]

with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    supervised, validation = [
        db.execute("""
            SELECT id, body, recommendation_status, rating_band
            FROM read_csv(?, header=true, all_varchar=true)
            ORDER BY substr(sha256('2026|' || id), 1, 16), id
            LIMIT ?
        """, [str(splits / f"{name}.csv"), size]).df()
        for name, size in [
            ("train_supervised", train_rows), ("validation", validation_rows),
        ]
    ]

assert len(supervised) == train_rows and len(validation) == validation_rows
assert not pseudo["id"].isin(supervised["id"]).any()

pseudo = pseudo.drop(columns="confidence")
pseudo["rating_band"] = pd.NA
combined = pd.concat([supervised, pseudo], ignore_index=True)
assert combined["id"].is_unique

semi_model = deepcopy(model)
teacher = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
semi_model.load_state_dict(teacher["parameters"], strict=False)
epoch = teacher["epoch"] + 1
seed = settings["seed"] + epoch
torch.manual_seed(seed)

weights = tuple(
    torch.tensor(settings[key], dtype=torch.float32, device="cuda")
    for key in ["recommendation_weights", "rating_band_weights"]
)
head_parameters = (
    list(semi_model.recommendation_head.parameters())
    + list(semi_model.rating_head.parameters())
)
optimizer = torch.optim.AdamW([
    {"params": semi_model.encoder.encoder.layer[-1].parameters(),
     "lr": settings["encoder_lr"]},
    {"params": head_parameters, "lr": settings["head_lr"]},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")

max_length, batch_size = settings["max_length"], settings["batch_size"]
train_loader = make_loader(
    combined, tokenizer, max_length, batch_size, shuffle=True, seed=seed,
)
validation_loader = make_loader(validation, tokenizer, max_length, batch_size)
combined_rows = len(combined)
del combined, supervised, pseudo, validation

print(
    f"Training: {train_rows:,} supervised + {pseudo_report['accepted_rows']:,} pseudo "
    f"= {combined_rows:,} | Validation: {validation_rows:,}",
    flush=True,
)
started = perf_counter()
train_metrics = run_epoch(
    semi_model, train_loader, weights, optimizer=optimizer, scaler=scaler,
)
validation_metrics = run_epoch(semi_model, validation_loader, weights)
assert all(math.isfinite(value) for value in [train_metrics["loss"], *validation_metrics.values()])

semi_checkpoint = output_dir / "trained.pt"
temporary = semi_checkpoint.with_suffix(".tmp")
torch.save({
    "parameters": {
        name: parameter.detach().cpu()
        for name, parameter in semi_model.named_parameters() if parameter.requires_grad
    },
    "epoch": epoch,
    "parent_checkpoint_sha256": pseudo_report["checkpoint_sha256"],
}, temporary)
temporary.replace(semi_checkpoint)

semi_report = {
    "parent_epoch": teacher["epoch"],
    "additional_epochs": 1,
    "seed": seed,
    "settings": settings,
    "supervised_rows": train_rows,
    "pseudo_labeled_rows": pseudo_report["accepted_rows"],
    "total_training_rows": combined_rows,
    "pseudo_threshold": pseudo_report["threshold"],
    "pseudo_targets_used": ["recommendation_status"],
    "train": train_metrics,
    "validation": validation_metrics,
    "seconds": round(perf_counter() - started, 1),
}
(reports / "parsbert_semi_supervised_1200k_training.json").write_text(
    json.dumps(semi_report, indent=2), encoding="utf-8",
)

print("\nSame 20,000 validation reviews")
for key in ["recommendation_macro_f1", "rating_macro_f1"]:
    before, after = training["validation"][key], validation_metrics[key]
    print(f"{key}: {before:.4f} -> {after:.4f} ({after - before:+.4f})")
print(f"Saved semi-supervised checkpoint and report | Elapsed: {semi_report['seconds']}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training: 1,200,000 supervised + 279,363 pseudo = 1,479,363 | Validation: 20,000


Train:   0%|          | 0/23116 [00:00<?, ?it/s]

Validation:   0%|          | 0/313 [00:00<?, ?it/s]


Same 20,000 validation reviews
recommendation_macro_f1: 0.6638 -> 0.6657 (+0.0019)
rating_macro_f1: 0.5718 -> 0.5769 (+0.0051)
Saved semi-supervised checkpoint and report | Elapsed: 1986.0s


In [ ]:
import json
import shutil
from pathlib import Path

import duckdb

root = Path("/content/drive/MyDrive/digikala-project-3")
reports = root / "reports"
output_dir = root / "artifacts" / "parsbert_semi_supervised_1200k"
assert (output_dir / "trained.pt").is_file(), "Complete semi-supervised training first."

pseudo_report = json.loads(
    (reports / "pseudo_labeling_1200k.json").read_text(encoding="utf-8")
)
reference_db = Path("/content/digikala_project.duckdb")
if not reference_db.is_file():
    temporary = reference_db.with_suffix(".tmp")
    shutil.copyfile(root / reference_db.name, temporary)
    temporary.replace(reference_db)

with duckdb.connect(str(reference_db), read_only=True) as db:
    groups = db.execute("""
        SELECT
            p.recommendation_status AS predicted,
            r.recommendation_status AS actual,
            m.id IS NOT NULL AS in_mask,
            count(*) AS n
        FROM read_csv(?, header=true, all_varchar=true) AS p
        LEFT JOIN masked_train_ids AS m ON p.id = m.id
        LEFT JOIN clean_reviews AS r ON p.id = r.id
        GROUP BY 1, 2, 3
    """, [str(output_dir / "pseudo_labeled.csv")]).df()

assert groups["in_mask"].all()
assert groups["actual"].notna().all()
assert groups["n"].sum() == pseudo_report["accepted_rows"]

classes = ["recommended", "not_recommended", "no_idea"]
matrix = (
    groups.pivot(index="actual", columns="predicted", values="n")
    .reindex(index=classes, columns=classes)
    .fillna(0).astype(int)
)

accepted = int(matrix.to_numpy().sum())
correct = int(matrix.to_numpy().trace())
by_prediction = {}

print(f"Correct pseudo-labels: {correct:,} / {accepted:,} ({correct / accepted:.2%})")
for label in classes:
    total = int(matrix[label].sum())
    correct_for_label = int(matrix.loc[label, label])
    precision = correct_for_label / total if total else None
    by_prediction[label] = {
        "accepted": total, "correct": correct_for_label, "precision": precision,
    }
    print(f"{label:16s} {correct_for_label:,} / {total:,} ({precision:.2%})")

print("\nRows: original label | Columns: pseudo-label")
print(matrix.to_string())

audit = {
    "threshold": pseudo_report["threshold"],
    "audited_rows": accepted,
    "correct_rows": correct,
    "accuracy": correct / accepted,
    "by_predicted_class": by_prediction,
    "confusion_matrix": {"class_order": classes, "counts": matrix.to_numpy().tolist()},
}
(reports / "pseudo_label_accuracy_1200k.json").write_text(
    json.dumps(audit, indent=2), encoding="utf-8",
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Correct pseudo-labels: 274,266 / 279,363 (98.18%)
recommended      253,468 / 256,238 (98.92%)
not_recommended  20,437 / 22,594 (90.45%)
no_idea          361 / 531 (67.98%)

Rows: original label | Columns: pseudo-label
predicted        recommended  not_recommended  no_idea
actual                                                
recommended           253468              630      153
not_recommended          509            20437       17
no_idea                 2261             1527      361


790

In [ ]:
import pandas as pd
import torch

from digikala_evaluation import evaluate_csv

checkpoint = torch.load(
    output_dir / "trained.pt", map_location="cpu", weights_only=True,
)
semi_model.load_state_dict(checkpoint["parameters"], strict=False)

supervised_validation = json.loads(
    (reports / "parsbert_supervised_1200k_validation.json").read_text(encoding="utf-8")
)

results = evaluate_csv(
    semi_model,
    splits / "validation.csv",
    tokenizer,
    settings["max_length"],
    expected_rows=supervised_validation["total_rows"],
)
results.update({"split": "validation", "checkpoint_epoch": checkpoint["epoch"]})

for target, record in results["heads"].items():
    reference = supervised_validation["heads"][target]
    assert record["evaluated_rows"] == reference["evaluated_rows"], target
    before, after = reference["metrics"], record["metrics"]
    old_f1 = before["macro avg"]["f1-score"]
    new_f1 = after["macro avg"]["f1-score"]
    names = record["confusion_matrix"]["class_order"]

    print(f"\n{target} | evaluated rows: {record['evaluated_rows']:,}")
    print(f"Accuracy: {before['accuracy']:.4f} -> {after['accuracy']:.4f}")
    print(f"Macro F1: {old_f1:.4f} -> {new_f1:.4f} ({new_f1 - old_f1:+.4f})")
    print(pd.DataFrame(after).T.loc[names + ["macro avg"]].round(4).to_string())
    if target == "rating_band":
        old_distance = reference["mean_absolute_ordinal_distance"]
        new_distance = record["mean_absolute_ordinal_distance"]
        print(f"Mean absolute ordinal distance: {old_distance:.4f} -> {new_distance:.4f}")

path = reports / "parsbert_semi_supervised_1200k_validation.json"
path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"\nSaved: {path.name}")

Evaluation:   0%|          | 0/38 [00:00<?, ?it/s]


recommendation_status | evaluated rows: 789,770
Accuracy: 0.7974 -> 0.8001
Macro F1: 0.6696 -> 0.6715 (+0.0018)
                 precision  recall  f1-score   support
recommended         0.9643  0.8333    0.8940  624568.0
not_recommended     0.6246  0.7959    0.6999   75961.0
no_idea             0.3327  0.5713    0.4205   89241.0
macro avg           0.6405  0.7335    0.6715  789770.0

rating_band | evaluated rows: 925,196
Accuracy: 0.6701 -> 0.6638
Macro F1: 0.5754 -> 0.5759 (+0.0005)
           precision  recall  f1-score   support
low           0.5272  0.5255    0.5264  163553.0
medium        0.3634  0.4914    0.4178  159968.0
high          0.8237  0.7473    0.7836  601675.0
macro avg     0.5714  0.5881    0.5759  925196.0
Mean absolute ordinal distance: 0.4322 -> 0.4356

Saved: parsbert_semi_supervised_1200k_validation.json


In [1]:
%pip -q install "transformers==5.22.1" duckdb

import hashlib
import json
import shutil
from pathlib import Path

import duckdb
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/digikala-project-3")
reports_dir = root / "reports"
original_dir = root / "artifacts" / "parsbert_supervised"
parent_dir = root / "artifacts" / "parsbert_supervised_1200k"
semi_dir = root / "artifacts" / "parsbert_semi_supervised_1200k"

def file_sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

parent_hash = file_sha256(parent_dir / "best.pt")
pseudo_report = json.loads(
    (reports_dir / "pseudo_labeling_1200k.json").read_text(encoding="utf-8")
)
semi_checkpoint = torch.load(
    semi_dir / "trained.pt", map_location="cpu", weights_only=True,
)
assert pseudo_report["checkpoint_sha256"] == parent_hash
assert semi_checkpoint["parent_checkpoint_sha256"] == parent_hash
assert pseudo_report["threshold"] == 0.90
del semi_checkpoint

reference_db = Path("/content/digikala_project.duckdb")
if not reference_db.is_file():
    print("Restoring reference database...", flush=True)
    temporary = reference_db.with_suffix(".tmp")
    shutil.copyfile(root / reference_db.name, temporary)
    temporary.replace(reference_db)

with duckdb.connect(
    str(reference_db),
    read_only=True,
    config={"memory_limit": "3GB", "threads": "2"},
) as db:
    groups = db.execute("""
        SELECT
            p.recommendation_status AS predicted,
            r.recommendation_status AS actual,
            m.id IS NOT NULL AS in_mask,
            count(*) AS n
        FROM read_csv(?, header=true, all_varchar=true) p
        LEFT JOIN masked_train_ids m ON p.id = m.id
        LEFT JOIN clean_reviews r ON p.id = r.id
        GROUP BY 1, 2, 3
    """, [str(semi_dir / "pseudo_labeled.csv")]).df()

assert groups["in_mask"].all()
assert groups["actual"].notna().all()
assert groups["n"].sum() == pseudo_report["accepted_rows"]

classes = ["recommended", "not_recommended", "no_idea"]
assert groups["actual"].isin(classes).all()
assert groups["predicted"].isin(classes).all()

matrix = (
    groups.pivot_table(
        index="actual", columns="predicted",
        values="n", aggfunc="sum", fill_value=0,
    )
    .reindex(index=classes, columns=classes, fill_value=0)
    .astype(int)
)
accepted = int(matrix.to_numpy().sum())
correct = int(matrix.to_numpy().trace())
by_prediction = {}

for label in classes:
    total = int(matrix[label].sum())
    correct_count = int(matrix.loc[label, label])
    by_prediction[label] = {
        "accepted": total,
        "correct": correct_count,
        "precision": correct_count / total if total else None,
    }

audit = {
    "parent_checkpoint_sha256": parent_hash,
    "threshold": 0.90,
    "audited_rows": accepted,
    "correct_rows": correct,
    "accuracy": correct / accepted,
    "by_predicted_class": by_prediction,
    "confusion_matrix": {
        "class_order": classes,
        "counts": matrix.to_numpy().tolist(),
    },
}
audit_path = reports_dir / "pseudo_label_accuracy_1200k.json"
temporary = audit_path.with_suffix(".tmp")
temporary.write_text(json.dumps(audit, indent=2), encoding="utf-8")
temporary.replace(audit_path)

print(f"Correct accepted pseudo-labels: {correct:,} / {accepted:,}")
print(f"Pseudo-label accuracy: {correct / accepted:.2%}")
print(pd.DataFrame(by_prediction).T.round(4).to_string())
print("\nRows: original label | Columns: pseudo-label")
print(matrix.to_string())
print(f"\nSaved: {audit_path.name}")

Mounted at /content/drive
Restoring reference database...
Correct accepted pseudo-labels: 274,266 / 279,363
Pseudo-label accuracy: 98.18%
                 accepted   correct  precision
recommended      256238.0  253468.0     0.9892
not_recommended   22594.0   20437.0     0.9045
no_idea             531.0     361.0     0.6798

Rows: original label | Columns: pseudo-label
predicted        recommended  not_recommended  no_idea
actual                                                
recommended           253468              630      153
not_recommended          509            20437       17
no_idea                 2261             1527      361

Saved: pseudo_label_accuracy_1200k.json


In [2]:
import tarfile

from transformers import AutoModel, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select T4 GPU in Runtime > Change runtime type.")

splits_dir = Path("/content/digikala_splits")
needed = ["test.csv", "split_manifest.json"]
if not all((splits_dir / name).is_file() for name in needed):
    print("Restoring test files...", flush=True)
    bundle = Path("/content/digikala_inputs.tar.gz")
    shutil.copyfile(root / bundle.name, bundle)
    with tarfile.open(bundle, "r:gz") as archive:
        archive.extractall("/content", filter="data")

for name in [
    "digikala_transformer.py",
    "digikala_training.py",
    "digikala_evaluation.py",
    "normalize_persian.py",
]:
    shutil.copyfile(original_dir / name, Path("/content") / name)

from digikala_transformer import DigikalaTransformer
from digikala_evaluation import evaluate_csv

training = json.loads(
    (reports_dir / "parsbert_supervised_1200k.json").read_text(encoding="utf-8")
)
settings = training["settings"]
assert settings["train_rows"] == 1_200_000

manifest = json.loads(
    (splits_dir / "split_manifest.json").read_text(encoding="utf-8")
)
expected_rows = manifest["exported_file_rows"]["test.csv"]

jobs = [
    (
        "supervised",
        parent_dir / "best.pt",
        "parsbert_supervised_1200k_test.json",
    ),
    (
        "semi_supervised",
        semi_dir / "trained.pt",
        "parsbert_semi_supervised_1200k_test.json",
    ),
]
test_results = {}
evaluation_model = None

for name, checkpoint_path, filename in jobs:
    report_path = reports_dir / filename
    checkpoint_hash = file_sha256(checkpoint_path)

    if report_path.is_file():
        result = json.loads(report_path.read_text(encoding="utf-8"))
        assert result["checkpoint_sha256"] == checkpoint_hash
        assert result["total_rows"] == expected_rows
        print(f"{name}: saved test results loaded.", flush=True)
    else:
        if evaluation_model is None:
            tokenizer = AutoTokenizer.from_pretrained(
                str(original_dir / "tokenizer"),
                local_files_only=True,
            )
            encoder = AutoModel.from_pretrained(
                settings["model"],
                revision=settings["revision"],
                add_pooling_layer=False,
                use_safetensors=False,
                weights_only=True,
                trust_remote_code=False,
            )
            evaluation_model = DigikalaTransformer(encoder).to("cuda")

        checkpoint = torch.load(
            checkpoint_path, map_location="cpu", weights_only=True,
        )
        assert set(checkpoint["parameters"]) == {
            key for key, parameter in evaluation_model.named_parameters()
            if parameter.requires_grad
        }
        evaluation_model.load_state_dict(checkpoint["parameters"], strict=False)
        evaluation_model.eval()

        print(f"\nEvaluating {name}: {expected_rows:,} test reviews...", flush=True)
        result = evaluate_csv(
            evaluation_model,
            splits_dir / "test.csv",
            tokenizer,
            settings["max_length"],
            expected_rows=expected_rows,
        )
        result.update({
            "split": "test",
            "model": name,
            "checkpoint_sha256": checkpoint_hash,
            "checkpoint_epoch": checkpoint["epoch"],
            "supervised_training_rows": 1_200_000,
        })

        temporary = report_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(result, indent=2), encoding="utf-8")
        temporary.replace(report_path)
        print(f"Saved: {filename}", flush=True)

    test_results[name] = result

def main_scores(record):
    metrics = record["metrics"]
    scores = {
        "accuracy": metrics["accuracy"],
        "macro_precision": metrics["macro avg"]["precision"],
        "macro_recall": metrics["macro avg"]["recall"],
        "macro_f1": metrics["macro avg"]["f1-score"],
        "balanced_accuracy": record["balanced_accuracy"],
    }
    if "mean_absolute_ordinal_distance" in record:
        scores["ordinal_distance"] = record["mean_absolute_ordinal_distance"]
    return scores

comparison = {
    "split": "test",
    "supervised_training_rows": 1_200_000,
    "pseudo_labeled_rows": pseudo_report["accepted_rows"],
    "pseudo_threshold": 0.90,
    "heads": {},
}

for target in ["recommendation_status", "rating_band"]:
    before = test_results["supervised"]["heads"][target]
    after = test_results["semi_supervised"]["heads"][target]
    assert before["evaluated_rows"] == after["evaluated_rows"]

    table = pd.DataFrame({
        "supervised": main_scores(before),
        "semi_supervised": main_scores(after),
    })
    table["change"] = table["semi_supervised"] - table["supervised"]
    comparison["heads"][target] = table.to_dict(orient="index")

    print(f"\nTEST COMPARISON — {target}")
    print(table.round(4).to_string())

    class_order = after["confusion_matrix"]["class_order"]
    print("\nSemi-supervised per-class metrics")
    print(
        pd.DataFrame(after["metrics"]).T
        .loc[class_order]
        .round(4).to_string()
    )

comparison_path = reports_dir / "semi_supervised_test_comparison.json"
temporary = comparison_path.with_suffix(".tmp")
temporary.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
temporary.replace(comparison_path)
print(f"\nSaved: {comparison_path.name}")

Restoring test files...


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

... (3 outputs left out)

Evaluation:   0%|          | 0/37 [00:00<?, ?it/s]

Saved: parsbert_supervised_1200k_test.json

Evaluating semi_supervised: 919,675 test reviews...


Evaluation:   0%|          | 0/37 [00:00<?, ?it/s]

Saved: parsbert_semi_supervised_1200k_test.json

TEST COMPARISON — recommendation_status
                   supervised  semi_supervised  change
accuracy               0.7991           0.8016  0.0025
macro_precision        0.6406           0.6396 -0.0010
macro_recall           0.7292           0.7328  0.0037
macro_f1               0.6692           0.6708  0.0016
balanced_accuracy      0.7292           0.7328  0.0037

Semi-supervised per-class metrics
                 precision  recall  f1-score   support
recommended         0.9645  0.8354    0.8953  623237.0
not_recommended     0.6221  0.7935    0.6974   74456.0
no_idea             0.3323  0.5696    0.4197   88083.0

TEST COMPARISON — rating_band
                   supervised  semi_supervised  change
accuracy               0.6707           0.6645 -0.0062
macro_precision        0.5692           0.5696  0.0004
macro_recall           0.5821           0.5869  0.0048
macro_f1               0.5736           0.5744  0.0007
balanced_accuracy   

In [3]:
import json
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

from digikala_training import make_loader
from digikala_transformer import REC_LABELS, BAND_LABELS
from normalize_persian import normalize_text

root = Path("/content/drive/MyDrive/digikala-project-3")
reports_dir = root / "reports"
artifacts_dir = root / "artifacts"
splits_dir = Path("/content/digikala_splits")
checkpoint_path = artifacts_dir / "parsbert_supervised_3000k" / "best.pt"

audit_path = reports_dir / "pseudo_label_accuracy_1200k.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
print(f"Accepted pseudo-label accuracy: {audit['accuracy']:.2%}")
print(pd.DataFrame(audit["by_predicted_class"]).T.round(4).to_string())

# Select a reproducible sample from the test set.
SAMPLE_ROWS = 20_000
SAMPLE_SEED = "error-audit-2026"

with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    sample = db.execute("""
        SELECT id, body, recommendation_status, rating_band
        FROM read_csv(?, header=true, all_varchar=true)
        ORDER BY substr(sha256(? || '|' || id), 1, 16), id
        LIMIT ?
    """, [
        str(splits_dir / "test.csv"), SAMPLE_SEED, SAMPLE_ROWS
    ]).df()

assert len(sample) == SAMPLE_ROWS and sample["id"].is_unique

# Restore the selected 3M model into the encoder used in the preceding cell.
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True,
)
assert set(checkpoint["parameters"]) == {
    name for name, parameter in evaluation_model.named_parameters()
    if parameter.requires_grad
}
evaluation_model.load_state_dict(checkpoint["parameters"], strict=False)
evaluation_model.eval()

loader = make_loader(
    sample, tokenizer,
    max_length=96,
    batch_size=256,
)
probabilities = [[], []]

with torch.inference_mode():
    for inputs, _, _ in tqdm(loader, desc="Predicting error-analysis sample"):
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = evaluation_model(**inputs.to("cuda"))
        for head, logits in enumerate(outputs):
            probabilities[head].append(
                logits.float().softmax(dim=1).cpu().numpy()
            )

for head, (target, encoding) in enumerate([
    ("recommendation_status", REC_LABELS),
    ("rating_band", BAND_LABELS),
]):
    scores = np.concatenate(probabilities[head])
    names = {code: label for label, code in encoding.items()}
    sample[f"transformer_{target}"] = [
        names[int(code)] for code in scores.argmax(axis=1)
    ]
    sample[f"confidence_{target}"] = scores.max(axis=1)
    sample[f"error_{target}"] = (
        sample[target].isin(encoding)
        & sample[target].ne(sample[f"transformer_{target}"])
    )

# Predict the same texts with the TF-IDF baseline.
texts = sample["body"].map(normalize_text)
vectorizer = joblib.load(artifacts_dir / "tfidf_vectorizer.joblib")
features = vectorizer.transform(texts)

for target in ["recommendation_status", "rating_band"]:
    bundle = joblib.load(artifacts_dir / f"tfidf_{target}.joblib")
    sample[f"tfidf_{target}"] = [
        bundle["class_names"][int(code)]
        for code in bundle["model"].predict(features)
    ]

# Measure repetition and label variation in the full cleaned dataset.
reference_db = Path("/content/digikala_project.duckdb")
assert reference_db.is_file()

with duckdb.connect(
    str(reference_db), read_only=True,
    config={"memory_limit": "3GB", "threads": "2"},
) as db:
    db.register("sample_texts", sample[["body"]].drop_duplicates())
    repetition = db.execute("""
        SELECT body,
            count(*) AS clean_text_rows,
            count(DISTINCT recommendation_status) AS recommendation_classes,
            count(DISTINCT rating_band) AS rating_classes
        FROM clean_reviews SEMI JOIN sample_texts USING (body)
        GROUP BY body
    """).df()

sample = sample.merge(repetition, on="body", how="left", validate="many_to_one")

# Read raw ratings only for error-analysis metadata.
print("Preparing original ratings for boundary analysis...", flush=True)
raw_path = hf_hub_download(
    repo_id="RadeAI/Digikala_comments_products",
    filename="digikala-comments.csv",
    repo_type="dataset",
)

with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    db.register("sample_ids", sample[["id"]])
    rates = db.execute("""
        SELECT trim(r.id) AS id,
            min(try_cast(r.rate AS DOUBLE)) AS min_rate,
            max(try_cast(r.rate AS DOUBLE)) AS max_rate,
            count(DISTINCT try_cast(r.rate AS DOUBLE)) AS rate_versions
        FROM read_csv(?, header=true, all_varchar=true) r
        SEMI JOIN sample_ids s ON trim(r.id) = s.id
        GROUP BY trim(r.id)
    """, [raw_path]).df()

sample = sample.merge(rates, on="id", how="left", validate="one_to_one")
assert sample["rate_versions"].notna().all()

# These flags guide selection. Their meaning requires reading each review.
sample["short_text"] = texts.str.count(r"\S+").le(3).to_numpy()
sample["negation"] = texts.str.contains(
    r"نیست|نبود|نمی|نخر|ندار|نشد|نکن|ناراضی", regex=True
).to_numpy()
sample["mixed_opinion"] = texts.str.contains(
    r"ولی|اما|با این حال|از طرفی", regex=True
).to_numpy()
sample["spelling_variant"] = sample["body"].str.contains(
    r"[يك]|میخوام|نمیشه|نمیارزه|میارزه", regex=True
)
sample["repeated_text"] = sample["clean_text_rows"].gt(1)
sample["conflicting_labels"] = (
    sample["recommendation_classes"].gt(1)
    | sample["rating_classes"].gt(1)
)
sample["rating_boundary"] = (
    sample["rate_versions"].eq(1)
    & sample["min_rate"].between(0, 5)
    & (
        sample["min_rate"].sub(2.5).abs().le(0.25)
        | sample["min_rate"].sub(4.0).abs().le(0.25)
    )
)
sample["model_disagreement"] = (
    sample["transformer_recommendation_status"].ne(
        sample["tfidf_recommendation_status"]
    )
    | sample["transformer_rating_band"].ne(sample["tfidf_rating_band"])
)
sample["possible_sarcasm"] = texts.str.contains(
    r"به[ ‌]*به|عجب|چه\s+(?:عالی|خوب)|شوخی", regex=True
).to_numpy()

errors = sample.loc[
    sample["error_recommendation_status"] | sample["error_rating_band"]
].copy()

flags = [
    "short_text", "negation", "mixed_opinion", "spelling_variant",
    "repeated_text", "conflicting_labels", "rating_boundary",
    "model_disagreement", "possible_sarcasm",
]
selected = []

# Include errors from each actual target class first.
for target, encoding in [
    ("recommendation_status", REC_LABELS),
    ("rating_band", BAND_LABELS),
]:
    for label in encoding:
        candidates = errors.index[
            errors[f"error_{target}"]
            & errors[target].eq(label)
            & ~errors.index.isin(selected)
        ]
        selected.extend(candidates[:1].tolist())

# Add cases with different review characteristics.
for flag in flags:
    candidates = errors.index[
        errors[flag] & ~errors.index.isin(selected)
    ]
    selected.extend(candidates[:2].tolist())

remaining = errors.index[~errors.index.isin(selected)]
selected.extend(remaining[:max(0, 24 - len(selected))].tolist())
cases = errors.loc[selected[:24]].copy()
assert len(cases) == 24

cases.insert(0, "case_number", range(1, len(cases) + 1))
cases.to_csv(
    reports_dir / "error_cases_for_review.csv",
    index=False, encoding="utf-8-sig",
)

selection_report = {
    "split": "test",
    "sample_rows": SAMPLE_ROWS,
    "sample_seed": SAMPLE_SEED,
    "sample_rule": "First rows ordered by SHA-256(seed + '|' + id) prefix, then id",
    "selected_cases": len(cases),
    "selection": "Purposeful coverage of actual classes and candidate error flags",
    "flags_require_manual_review": True,
}
(reports_dir / "error_selection.json").write_text(
    json.dumps(selection_report, indent=2), encoding="utf-8",
)

for _, row in cases.iterrows():
    active_flags = ", ".join(flag for flag in flags if row[flag])
    print(f"\nCASE {row['case_number']} | ID: {row['id']}")
    print(f"Selection flags: {active_flags}")
    print(f"Text: {row['body']}")
    print(
        "Recommendation — "
        f"actual: {row['recommendation_status'] or 'missing'} | "
        f"ParsBERT: {row['transformer_recommendation_status']} | "
        f"TF-IDF: {row['tfidf_recommendation_status']}"
    )
    print(
        "Rating band — "
        f"actual: {row['rating_band'] or 'missing'} | "
        f"ParsBERT: {row['transformer_rating_band']} | "
        f"TF-IDF: {row['tfidf_rating_band']}"
    )
    print(
        f"Raw rate range: {row['min_rate']:g}–{row['max_rate']:g} | "
        f"Same-text rows: {int(row['clean_text_rows']):,} | "
        f"Recommendation classes: {int(row['recommendation_classes'])} | "
        f"Rating classes: {int(row['rating_classes'])}"
    )

print("\nSaved: error_cases_for_review.csv and error_selection.json")

Accepted pseudo-label accuracy: 98.18%
                 accepted   correct  precision
recommended      256238.0  253468.0     0.9892
not_recommended   22594.0   20437.0     0.9045
no_idea             531.0     361.0     0.6798


Predicting error-analysis sample:   0%|          | 0/79 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Preparing original ratings for boundary analysis...


digikala-comments.csv: reconstructing file:   0%|          |  0.00B / 1.28GB            

digikala-comments.csv: downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


CASE 1 | ID: 51046284
Selection flags: short_text, model_disagreement
Text: خیلی متوسط هست 
Recommendation — actual: recommended | ParsBERT: no_idea | TF-IDF: not_recommended
Rating band — actual: medium | ParsBERT: medium | TF-IDF: medium
Raw rate range: 3–3 | Same-text rows: 1 | Recommendation classes: 1 | Rating classes: 1

CASE 2 | ID: 34264659
Selection flags: negation, model_disagreement
Text: نمیدونم دست دومه یا هرچی 1 پر از خط و خش بود 2 انگار سوزونده بودنش دیسک آب شده بود
Recommendation — actual: not_recommended | ParsBERT: no_idea | TF-IDF: not_recommended
Rating band — actual: low | ParsBERT: low | TF-IDF: low
Raw rate range: 1–1 | Same-text rows: 1 | Recommendation classes: 1 | Rating classes: 1

CASE 3 | ID: 43279550
Selection flags: 
Text: خشک و غیر قابل انعطاف ضایع
Recommendation — actual: no_idea | ParsBERT: not_recommended | TF-IDF: not_recommended
Rating band — actual: low | ParsBERT: low | TF-IDF: low
Raw rate range: 1–1 | Same-text rows: 1 | Recommendation classes:

In [4]:
import re
from pathlib import Path

import pandas as pd

reports_dir = Path("/content/drive/MyDrive/digikala-project-3/reports")
cases_path = reports_dir / "error_cases_for_review.csv"
reviewed = pd.read_csv(
    cases_path, dtype={"id": str}, keep_default_na=False,
)

notes = {
    "51046284": (
        "متن کوتاه و توصیهٔ مبهم",
        "عبارت «خیلی متوسط» ارزیابی میانه‌ای بیان می‌کند. کاربر خرید را پیشنهاد کرده، "
        "مدل گزینهٔ بی‌نظر را پیش‌بینی کرده و بازهٔ امتیاز را درست تشخیص داده است."
    ),
    "34264659": (
        "عدم قطعیت در کنار شرح عیب",
        "متن با «نمیدونم» شروع می‌شود و سپس خط‌وخش و آب‌شدن دیسک را شرح می‌دهد. "
        "مدل پیشنهاد خرید را بی‌نظر تشخیص داده است. مدل پایه پیشنهاد خرید و هر دو مدل بازهٔ امتیاز را درست پیش‌بینی کرده‌اند."
    ),
    "43279550": (
        "تفاوت احساس منفی با گزینهٔ پیشنهاد خرید",
        "متن صریحاً منفی است و هر دو مدل خرید را پیشنهادنشده پیش‌بینی کرده‌اند. "
        "کاربر گزینهٔ بی‌نظر را ثبت کرده است. بازهٔ امتیاز پایین درست تشخیص داده شده است."
    ),
    "25761394": (
        "ناسازگاری متن مثبت و امتیاز ثبت‌شده",
        "کاربر چسب‌داربودن کالا را مثبت توصیف کرده و خرید را پیشنهاد داده است. "
        "امتیاز ثبت‌شده صفر و در بازهٔ پایین است. هر دو مدل از متن، بازهٔ بالا را پیش‌بینی کرده‌اند."
    ),
    "17937444": (
        "نظر منفی با امتیاز متوسط",
        "نظر بر بسته‌بندی آسیب‌دیده و نامناسب‌بودن برای هدیه تمرکز دارد. "
        "هر دو مدل پیشنهادنکردن خرید را درست تشخیص داده‌اند، ولی امتیاز ۳ را در بازهٔ پایین پیش‌بینی کرده‌اند."
    ),
    "23313616": (
        "خطای شدید بازهٔ امتیاز و متن تکراری",
        "متن «کتاب خوبی بود راضی بودم» و امتیاز ۴٫۳۵ با بازهٔ بالا همسو هستند. "
        "ترنسفورمر بازهٔ پایین را پیش‌بینی کرده و مدل پایه درست پاسخ داده است. "
        "همین متن در ۱۴ رکورد و هر سه بازهٔ امتیاز دیده شده است."
    ),
    "33359536": (
        "متن تک‌واژه‌ای و برچسب‌های متفاوت",
        "واژهٔ «معمولی» با امتیاز ۲ ثبت شده است. هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. "
        "این متن ۱۳٬۱۸۰ بار و با هر سه کلاسِ هر دو هدف دیده شده است."
    ),
    "43349539": (
        "ارزش خرید نسبی و مرز بازهٔ امتیاز",
        "عبارت «به قیمتش می‌ارزه» رضایت نسبی را بیان می‌کند. امتیاز ۴ طبق تعریف در بازهٔ بالاست. "
        "ترنسفورمر بازهٔ متوسط را پیش‌بینی کرده و مدل پایه درست پاسخ داده است."
    ),
    "13644798": (
        "نفی در عبارت مثبتِ محدود",
        "«بد نیست» در کنار مقایسهٔ نرمی برس، رضایت محدود را بیان می‌کند. "
        "ترنسفورمر پیشنهاد خرید را بی‌نظر پیش‌بینی کرده است. مدل پایه پیشنهاد خرید را درست تشخیص داده "
        "و هر دو مدل بازهٔ امتیاز را درست پیش‌بینی کرده‌اند."
    ),
    "6805470": (
        "نظر ترکیبی و اختلاف متن با پیشنهاد ثبت‌شده",
        "ابتدای متن دربارهٔ بوی عطر مثبت است و جملهٔ پایانی می‌گوید ارزش خرید ندارد. "
        "کاربر خرید را پیشنهاد کرده و امتیاز ۳٫۳ داده است. پیش‌بینی ترنسفورمر با جملهٔ پایانی همسو است "
        "و با هر دو برچسب ثبت‌شده اختلاف دارد."
    ),
    "29087639": (
        "مزیت و عیب در یک نظر",
        "کاربر ماندگاری را خوب و بو را تیز توصیف کرده است. "
        "ترنسفورمر پیشنهاد خرید را بی‌نظر پیش‌بینی کرده، ولی کاربر خرید را پیشنهاد داده است. "
        "بازهٔ امتیاز هر دو مدل درست است."
    ),
    "10424313": (
        "ارزیابی متفاوت کتاب و ترجمه",
        "نظر شامل تحسین کتاب و انتقاد شدید از ترجمه است. کاربر خرید را پیشنهاد داده و امتیاز ۴ ثبت کرده است. "
        "ترنسفورمر بی‌نظر و بازهٔ متوسط را پیش‌بینی کرده است. مدل پایه نیز در هر دو هدف خطا دارد."
    ),
    "45275211": (
        "انتقاد از قیمت و تفاوت نوشتاری",
        "متن قیمت را نسبت به بیرون گران‌تر توصیف می‌کند و حرف عربی در آن دیده می‌شود. "
        "امتیاز ۲ در بازهٔ پایین است و هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. "
        "این نمونه در تحلیل خطای بازهٔ امتیاز حساب شده است."
    ),
    "39398883": (
        "توصیهٔ منفی برای کاربرد مشخص",
        "متن استفاده برای نوزاد را توصیه نمی‌کند و کاربر گزینهٔ بی‌نظر را ثبت کرده است. "
        "هر دو مدل پیشنهادنکردن خرید را پیش‌بینی کرده‌اند. ترنسفورمر بازهٔ امتیاز ۳ را درست تشخیص داده است."
    ),
    "19756647": (
        "متن مثبت، امتیاز صفر و تکرار",
        "متن «خوب و مناسب بود» با امتیاز صفر ثبت شده است. هر دو مدل بازهٔ بالا را پیش‌بینی کرده‌اند. "
        "این متن در ۴۲۹ رکورد و با هر سه کلاسِ هر دو هدف دیده شده است."
    ),
    "44415035": (
        "رضایت نسبت به قیمت",
        "کاربر «نسبت به قیمتش خوبه» نوشته و امتیاز ۵ داده است. هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. "
        "همین متن در ۸٬۰۱۵ رکورد و با هر سه کلاسِ هر دو هدف ثبت شده است."
    ),
    "45148985": (
        "ارزش خرید و امتیاز بالا",
        "متن ارزش خرید را تأیید می‌کند و امتیاز ۵ دارد. هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. "
        "این متن در ۲۱ رکورد و با هر سه بازهٔ امتیاز دیده شده است."
    ),
    "52286501": (
        "نظر بسیار کوتاه و شدت نامشخص رضایت",
        "عبارت «خوب بود» با امتیاز ۳ ثبت شده است. هر دو مدل بازهٔ بالا را پیش‌بینی کرده‌اند. "
        "این متن ۷۳٬۸۱۲ بار و با هر سه کلاسِ هر دو هدف دیده شده است."
    ),
    "31192420": (
        "رضایت مشروط به قیمت و کاربرد",
        "کاربر کالا را با توجه به قیمت و مصرف روزانه خوب دانسته و امتیاز ۴ داده است. "
        "هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. امتیاز دقیقاً روی مرز شروع بازهٔ بالا قرار دارد."
    ),
    "40294020": (
        "مرز امتیاز و چسبیدن واژه‌ها",
        "متن شامل نوشتار چسبیدهٔ «نسبتبه» و ارزیابی مثبت اقتصادی است. "
        "امتیاز ۴ در بازهٔ بالاست. ترنسفورمر متوسط و مدل پایه بالا را پیش‌بینی کرده‌اند. "
        "سهم تفاوت نوشتاری در این خطا از مشاهدهٔ یک نمونه مشخص نمی‌شود."
    ),
    "22231863": (
        "نتیجهٔ مثبت استفاده با امتیاز صفر",
        "کاربر نتیجهٔ ترکیب رنگ را خوب توصیف کرده و امتیاز صفر ثبت کرده است. "
        "ترنسفورمر بازهٔ بالا و مدل پایه بازهٔ پایین را پیش‌بینی کرده‌اند."
    ),
    "54064030": (
        "رضایت بیان‌شده با امتیاز متوسط",
        "جملهٔ «در کل بچه ها دوست دارن» مثبت است و امتیاز ثبت‌شده ۳ است. "
        "ترنسفورمر بازهٔ بالا را پیش‌بینی کرده و مدل پایه بازهٔ متوسط را درست تشخیص داده است."
    ),
    "34515850": (
        "نظر بلند و چندوجهی",
        "متن مزایا، محدودیت استفاده و قیمت بالا را شرح می‌دهد و در پایان خرید در تخفیف را مطلوب می‌داند. "
        "ترنسفورمر گزینهٔ بی‌نظر را درست پیش‌بینی کرده و امتیاز ۴ را متوسط تشخیص داده است. "
        "نشانهٔ خودکار کنایه از تطبیق عبارت داخل «خوبه به خصوص» ایجاد شده بود."
    ),
    "49445857": (
        "رضایت نسبی، مرز امتیاز و تکرار",
        "عبارت «خوبه به نسبت» با امتیاز ۴ ثبت شده و هر دو مدل بازهٔ متوسط را پیش‌بینی کرده‌اند. "
        "این متن در ۴۶ رکورد و با هر سه بازهٔ امتیاز دیده شده است. "
        "نشانهٔ خودکار کنایه از تطبیق عبارت داخل «خوبه به نسبت» ایجاد شده بود."
    ),
}

assert len(reviewed) == 24
assert set(reviewed["id"]) == set(notes)

# Require complete words when applying the selection flags.
texts = reviewed["body"].astype(str)
reviewed["mixed_opinion"] = texts.str.contains(
    r"(?<![\w\u200c])(?:ولی|اما|با این حال|از طرفی)(?![\w\u200c])",
    regex=True,
)
reviewed["possible_sarcasm"] = texts.str.contains(
    r"(?<![\w\u200c])(?:به[ ‌]+به|عجب|چه\s+(?:عالی|خوب)|شوخی)(?![\w\u200c])",
    regex=True,
)
reviewed["error_category"] = [notes[row_id][0] for row_id in reviewed["id"]]
reviewed["review_note"] = [notes[row_id][1] for row_id in reviewed["id"]]
reviewed["manually_reviewed"] = True
reviewed["sarcasm_confirmed"] = False
reviewed.to_csv(cases_path, index=False, encoding="utf-8-sig")

lines = [
    "> # تحلیل ۲۴ خطای مدل منتخب",
    ">",
    "> نمونه‌ها هدفمند از یک نمونهٔ ثابت ۲۰ هزارنفری آزمون انتخاب شده‌اند. "
    "تعداد نمونه‌های هر نوع خطا، فراوانی آن نوع خطا در کل داده را نشان نمی‌دهد.",
    ">",
    "> توضیح‌ها بر پایهٔ متن و برچسب ثبت‌شده نوشته شده‌اند و علت درونی تصمیم مدل را اثبات نمی‌کنند. "
    "امتیاز صفر طبق تعریف پروژه در بازهٔ پایین حفظ شده است.",
    ">",
]

for _, row in reviewed.iterrows():
    lines.extend([
        f"> ## نمونهٔ {int(row['case_number'])}",
        ">",
        f"> شناسهٔ `{row['id']}`",
        ">",
    ])
    for text_line in str(row["body"]).strip().splitlines():
        lines.append(f"> {text_line}")
    lines.extend([
        ">",
        f"> نوع خطا — {row['error_category']}",
        ">",
        f"> {row['review_note']}",
        ">",
    ])

report_path = reports_dir / "error_analysis.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Reviewed cases: {len(reviewed)}")
print("Saved: error_cases_for_review.csv and error_analysis.md")

Reviewed cases: 24
Saved: error_cases_for_review.csv and error_analysis.md


In [5]:
import duckdb
import numpy as np
import torch

from digikala_training import make_loader
from digikala_transformer import REC_LABELS, BAND_LABELS

# Retrieve additional candidates for manual sarcasm review.
with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    candidates = db.execute("""
        SELECT id, body, recommendation_status, rating_band
        FROM read_csv(
            '/content/digikala_splits/test.csv',
            header=true, all_varchar=true
        )
        WHERE
            regexp_matches(
                body,
                '(اگه|اگر).{0,80}پول.{0,60}(اضاف|زیاد|دور|حروم)'
            )
            OR regexp_matches(
                body,
                '(از بس|انقدر|اینقدر|این قدر).{0,20}(خوب|عالی)'
            )
            OR regexp_matches(
                body,
                '(عالی|شاهکار|بهترین).{0,100}(سطل|زباله|نخرید)'
            )
            OR contains(body, 'عجب کیفیت')
            OR contains(body, 'چه عالی')
        ORDER BY substr(sha256('sarcasm-audit-2026|' || id), 1, 16), id
        LIMIT 256
    """).df()

if candidates.empty:
    print("No matching sarcasm candidates found.")
else:
    predictions = [[], []]
    loader = make_loader(candidates, tokenizer, max_length=96, batch_size=256)
    evaluation_model.eval()

    with torch.inference_mode():
        for inputs, _, _ in loader:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = evaluation_model(**inputs.to("cuda"))
            for head, logits in enumerate(outputs):
                predictions[head].extend(logits.argmax(dim=1).cpu().tolist())

    for head, (target, encoding) in enumerate([
        ("recommendation_status", REC_LABELS),
        ("rating_band", BAND_LABELS),
    ]):
        names = {code: label for label, code in encoding.items()}
        candidates[f"predicted_{target}"] = [
            names[code] for code in predictions[head]
        ]
        candidates[f"error_{target}"] = (
            candidates[target].isin(encoding)
            & candidates[target].ne(candidates[f"predicted_{target}"])
        )

    errors = candidates.loc[
        candidates["error_recommendation_status"]
        | candidates["error_rating_band"]
    ].head(8)

    errors.to_csv(
        reports_dir / "sarcasm_candidates.csv",
        index=False, encoding="utf-8-sig",
    )

    print(f"Candidates inspected: {len(candidates)}")
    print(f"Misclassified candidates shown: {len(errors)}")

    for number, (_, row) in enumerate(errors.iterrows(), start=1):
        print(f"\nCANDIDATE {number} | ID: {row['id']}")
        print(f"Text: {row['body']}")
        print(
            f"Recommendation — actual: {row['recommendation_status'] or 'missing'}"
            f" | predicted: {row['predicted_recommendation_status']}"
        )
        print(
            f"Rating band — actual: {row['rating_band'] or 'missing'}"
            f" | predicted: {row['predicted_rating_band']}"
        )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidates inspected: 256
Misclassified candidates shown: 8

CANDIDATE 1 | ID: 4556881
Text: رنگیه که اگه حساسیت نداشته باشید پولتون حروم نشده و بسته به جنس مو با کاتالوگ هماهنگه
Recommendation — actual: recommended | predicted: no_idea
Rating band — actual: high | predicted: medium

CANDIDATE 2 | ID: 38963367
Text: فر نمیکردم جنسش اینقدر خوب باشه راضیم و پیشنهاد میکنم
Recommendation — actual: no_idea | predicted: recommended
Rating band — actual: low | predicted: high

CANDIDATE 3 | ID: 43140293
Text: انتخاب عالی برای نخریدن
افتضاح
Recommendation — actual: not_recommended | predicted: recommended
Rating band — actual: low | predicted: low

CANDIDATE 4 | ID: 26876483
Text: فکر نمیکردم اینقدر خوب و خوشگل باشه دوباره ازشون سفارش دادم.قرار دادنش خیلی راحته.حتما بخرید.من برا زیرش لاک پایه گذاشتم و حتما باید لاک کاملا خشک بشه.
Recommendation — actual: recommended | predicted: recommended
Rating band — actual: medium | predicted: high

CANDIDATE 5 | ID: 45340776
Text: خیلی کنسول عالی نسل نهم

In [6]:
# 1. Save the eight additional error analyses.

from pathlib import Path

import pandas as pd

root = Path("/content/drive/MyDrive/digikala-project-3")
reports_dir = root / "reports"
extra = pd.read_csv(
    reports_dir / "sarcasm_candidates.csv",
    dtype={"id": str},
    keep_default_na=False,
)

extra_notes = {
    "4556881": (
        "رضایت مشروط و ساختار منفی",
        "«پولتون حروم نشده» در کنار شرط حساسیت و جنس مو، ارزیابی مثبت مشروط است. "
        "مدل گزینهٔ بی‌نظر و امتیاز متوسط را پیش‌بینی کرده، در حالی که برچسب‌ها پیشنهاد خرید و امتیاز بالا هستند."
    ),
    "38963367": (
        "اختلاف متن مثبت با برچسب‌های ثبت‌شده",
        "متن رضایت و پیشنهاد خرید را صریح بیان می‌کند. پیش‌بینی مدل با این بیان مثبت همسو است، "
        "ولی برچسب ثبت‌شده بی‌نظر و بازهٔ امتیاز پایین است."
    ),
    "43140293": (
        "کنایهٔ روشن",
        "عبارت «انتخاب عالی برای نخریدن» با واژهٔ مثبت شروع می‌شود و معنای منفی دارد. "
        "کلمهٔ «افتضاح» هم این خوانش را تقویت می‌کند. مدل پیشنهاد خرید را اشتباه "
        "و بازهٔ امتیاز پایین را درست پیش‌بینی کرده است."
    ),
    "26876483": (
        "رضایت زیاد در متن و امتیاز متوسط",
        "متن شامل رضایت، سفارش دوباره و «حتما بخرید» است. مدل پیشنهاد خرید را درست تشخیص داده "
        "و بازهٔ بالا را پیش‌بینی کرده، در حالی که امتیاز ثبت‌شده متوسط است."
    ),
    "45340776": (
        "ارزیابی متفاوت کالا و خدمات جانبی",
        "متن کنسول را تحسین می‌کند و از کارت طلایی انتقاد می‌کند. پیشنهاد خرید از فروشگاه نیز مثبت است. "
        "مدل امتیاز پایین را پیش‌بینی کرده، در حالی که بازهٔ ثبت‌شده بالاست. "
        "برچسب پیشنهاد خرید این رکورد موجود نیست."
    ),
    "10606665": (
        "غافلگیری مثبت با امتیاز متوسط",
        "عبارت «فک نمی‌کردم» دربارهٔ انتظار قبلی است و ادامهٔ جمله رضایت از پوشش‌دهی را بیان می‌کند. "
        "مدل پیشنهاد خرید را درست و بازهٔ امتیاز را بالاتر از برچسب ثبت‌شده پیش‌بینی کرده است."
    ),
    "14966530": (
        "ارزیابی مثبت اقتصادی و محیط‌زیستی",
        "متن کالا را از نظر کاربرد، هزینه و محیط‌زیست تحسین می‌کند. "
        "مدل پیشنهاد خرید را درست پیش‌بینی کرده و بازهٔ بالا را به جای بازهٔ پایین ثبت‌شده انتخاب کرده است."
    ),
    "14544941": (
        "ابهام دستوری و عبارت منفی در توصیهٔ مثبت",
        "«خیلی عالی نخرید ضرر کردید» احتمالاً خریدن را توصیه می‌کند و ساختاری محاوره‌ای و فشرده دارد. "
        "مدل امتیاز پایین را پیش‌بینی کرده، در حالی که بازهٔ ثبت‌شده بالاست. "
        "این برداشت زبانی با ابهام همراه است."
    ),
}

assert set(extra["id"]) == set(extra_notes)
extra["error_category"] = [extra_notes[row_id][0] for row_id in extra["id"]]
extra["review_note"] = [extra_notes[row_id][1] for row_id in extra["id"]]
extra["manually_reviewed"] = True
extra["sarcasm_confirmed"] = extra["id"].eq("43140293")
extra.to_csv(
    reports_dir / "sarcasm_candidates.csv",
    index=False, encoding="utf-8-sig",
)

base_report = (reports_dir / "error_analysis.md").read_text(encoding="utf-8")
marker = "> ## بررسی تکمیلی نمونه‌های هدفمند"
base_report = base_report.split(marker)[0].rstrip()
lines = [
    base_report,
    ">",
    marker,
    ">",
    "> این هشت نمونه از میان ۲۵۶ نامزد هدفمند انتخاب شده‌اند. "
    "تعداد آن‌ها فراوانی کنایه در کل داده را نشان نمی‌دهد.",
    ">",
]

for number, (_, row) in enumerate(extra.iterrows(), start=25):
    lines.extend([
        f"> ### نمونهٔ {number}",
        ">",
        f"> شناسهٔ `{row['id']}`",
        ">",
    ])
    for text_line in row["body"].strip().splitlines():
        lines.append(f"> {text_line}")
    lines.extend([
        ">",
        f"> نوع خطا — {row['error_category']}",
        ">",
        f"> {row['review_note']}",
        ">",
    ])

(reports_dir / "error_analysis.md").write_text(
    "\n".join(lines), encoding="utf-8",
)
print("Error analysis saved: 32 reviewed cases, including a clear sarcasm error.")

Error analysis saved: 32 reviewed cases, including a clear sarcasm error.


In [7]:
# 2. Export the complete selected model and baseline artifacts.

import hashlib
import json
import shutil
import sys
from importlib.metadata import version

import torch
from safetensors.torch import save_file

selected_dir = root / "artifacts" / "parsbert_supervised_3000k"
release_dir = root / "release"
models_dir = release_dir / "models"
parsbert_dir = models_dir / "parsbert"
parsbert_dir.mkdir(parents=True, exist_ok=True)

training = json.loads(
    (reports_dir / "parsbert_supervised_3000k.json").read_text(encoding="utf-8")
)
test_report = json.loads(
    (reports_dir / "parsbert_supervised_3000k_test.json").read_text(encoding="utf-8")
)

checkpoint_path = selected_dir / "best.pt"
with checkpoint_path.open("rb") as file:
    checkpoint_hash = hashlib.file_digest(file, "sha256").hexdigest()
assert checkpoint_hash == test_report["checkpoint_sha256"]

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert set(checkpoint["parameters"]) == {
    name for name, parameter in evaluation_model.named_parameters()
    if parameter.requires_grad
}
evaluation_model.load_state_dict(checkpoint["parameters"], strict=False)
evaluation_model.eval()

print("Saving the complete trained encoder...", flush=True)
evaluation_model.encoder.save_pretrained(
    parsbert_dir / "encoder",
    safe_serialization=True,
)
tokenizer.save_pretrained(parsbert_dir / "tokenizer")

head_weights = {
    name: tensor.detach().cpu().contiguous()
    for name, tensor in evaluation_model.state_dict().items()
    if name.startswith(("recommendation_head.", "rating_head."))
}
assert len(head_weights) == 4
save_file(head_weights, str(parsbert_dir / "heads.safetensors"))

for filename in [
    "tfidf_vectorizer.joblib",
    "tfidf_recommendation_status.joblib",
    "tfidf_rating_band.joblib",
]:
    shutil.copyfile(root / "artifacts" / filename, models_dir / filename)

for filename in ["normalize_persian.py", "digikala_transformer.py"]:
    shutil.copyfile(selected_dir / filename, release_dir / filename)

packages = [
    "torch", "transformers", "safetensors", "numpy", "scikit-learn",
    "joblib", "pandas", "duckdb", "matplotlib", "tqdm", "huggingface-hub",
]
environment = {package: version(package) for package in packages}

config = {
    "model_version": "parsbert-3000k-v1",
    "checkpoint_sha256": checkpoint_hash,
    "max_length": training["settings"]["max_length"],
    "input_column": "body",
    "recommendation_labels": REC_LABELS,
    "rating_band_labels": BAND_LABELS,
    "python_version": sys.version.split()[0],
    "package_versions": environment,
}
(models_dir / "inference_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8",
)

requirements = [
    f"{package}=={package_version.split('+')[0]}"
    for package, package_version in environment.items()
]
(release_dir / "requirements.txt").write_text(
    "\n".join(requirements) + "\n", encoding="utf-8",
)
print("Complete offline model assets exported.")

Saving the complete trained encoder...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Complete offline model assets exported.


In [8]:
%%writefile /content/predict.py

import argparse
import json
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import torch
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer

from digikala_transformer import DigikalaTransformer
from normalize_persian import normalize_text


class ReviewPredictor:
    def __init__(self, project_dir, device="cpu"):
        self.device = torch.device(device)
        if self.device.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA is unavailable. Use --device cpu.")

        models_dir = Path(project_dir) / "models"
        parsbert_dir = models_dir / "parsbert"
        self.config = json.loads(
            (models_dir / "inference_config.json").read_text(encoding="utf-8")
        )
        self.vectorizer = joblib.load(models_dir / "tfidf_vectorizer.joblib")
        self.feature_names = self.vectorizer.get_feature_names_out()
        self.baselines = {
            target: joblib.load(models_dir / f"tfidf_{target}.joblib")
            for target in ["recommendation_status", "rating_band"]
        }
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(parsbert_dir / "tokenizer"), local_files_only=True,
        )
        encoder = AutoModel.from_pretrained(
            str(parsbert_dir / "encoder"),
            local_files_only=True,
            use_safetensors=True,
            trust_remote_code=False,
            add_pooling_layer=False,
        )
        self.model = DigikalaTransformer(encoder)
        heads = load_file(str(parsbert_dir / "heads.safetensors"))
        for name in ["recommendation_head", "rating_head"]:
            getattr(self.model, name).load_state_dict({
                "weight": heads[f"{name}.weight"],
                "bias": heads[f"{name}.bias"],
            })
        self.model.to(self.device).eval()

    def _baseline_result(self, features, bundle):
        classifier = bundle["model"]
        probabilities = classifier.predict_proba(features)[0]
        order = np.argsort(-probabilities, kind="stable")
        winner, runner_up = int(order[0]), int(order[1])
        labels = [
            bundle["class_names"][int(code)]
            for code in classifier.classes_
        ]

        # Exact feature contributions to the winning-vs-runner-up logit margin.
        coefficient_margin = classifier.coef_[winner] - classifier.coef_[runner_up]
        contributions = coefficient_margin[features.indices] * features.data
        strongest = np.argsort(-np.abs(contributions))[:5]
        terms = [{
            "term": str(self.feature_names[features.indices[index]]),
            "logit_margin_contribution": float(contributions[index]),
        } for index in strongest]

        return {
            "label": labels[winner],
            "probabilities": {
                label: float(probability)
                for label, probability in zip(labels, probabilities)
            },
            "explanation": {
                "runner_up": labels[runner_up],
                "bias_margin": float(
                    classifier.intercept_[winner] - classifier.intercept_[runner_up]
                ),
                "strongest_terms": terms,
                "positive_contribution_favors": labels[winner],
            },
        }

    def predict(self, text):
        started = perf_counter()
        normalized = normalize_text(text)
        if not normalized:
            raise ValueError("Text is empty after normalization.")

        features = self.vectorizer.transform([normalized])
        result = {
            "model_version": self.config["model_version"],
            "text": text,
            "normalized_text": normalized,
            "tfidf": {
                target: self._baseline_result(features, bundle)
                for target, bundle in self.baselines.items()
            },
            "parsbert": {},
        }

        original_ids = self.tokenizer(
            normalized, truncation=False, verbose=False,
        )["input_ids"]
        inputs = self.tokenizer(
            normalized,
            truncation=True,
            max_length=self.config["max_length"],
            return_tensors="pt",
        ).to(self.device)

        with torch.inference_mode():
            outputs = self.model(**inputs)

        for target, logits, encoding in zip(
            ["recommendation_status", "rating_band"],
            outputs,
            [
                self.config["recommendation_labels"],
                self.config["rating_band_labels"],
            ],
        ):
            probabilities = logits.float().softmax(dim=1)[0].cpu().numpy()
            names = sorted(encoding, key=encoding.get)
            result["parsbert"][target] = {
                "label": names[int(probabilities.argmax())],
                "probabilities": {
                    name: float(probabilities[encoding[name]])
                    for name in names
                },
            }

        result["token_count_before_truncation"] = len(original_ids)
        result["token_count_used"] = int(inputs["attention_mask"].sum().item())
        result["truncated"] = len(original_ids) > self.config["max_length"]
        result["latency_ms"] = round((perf_counter() - started) * 1000, 2)
        return result


def main():
    parser = argparse.ArgumentParser()
    choice = parser.add_mutually_exclusive_group(required=True)
    choice.add_argument("--text")
    choice.add_argument("--demo", action="store_true")
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    args = parser.parse_args()

    predictor = ReviewPredictor(Path(__file__).resolve().parent, args.device)
    texts = [
        "خوبه",
        "معمولی بود، هنوز مطمئن نیستم ارزش خرید داشته باشه.",
        "بعد از دو روز خراب شد و اصلاً پیشنهاد نمی‌کنم.",
        "کیفیت ساخت خوبه ولی باتری زود تموم میشه و قیمتش بالاست.",
    ] if args.demo else [args.text]

    try:
        results = [predictor.predict(text) for text in texts]
    except ValueError as error:
        parser.error(str(error))
    print(json.dumps(results if args.demo else results[0], ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

Writing /content/predict.py


In [9]:
# 4. Test the actual command-line script with offline loading on CPU.

import json
import os
import shutil
import subprocess
import sys

shutil.copyfile("/content/predict.py", release_dir / "predict.py")

offline_environment = {
    **os.environ,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}

print("Running four offline prediction examples on CPU...", flush=True)
process = subprocess.run(
    [sys.executable, str(release_dir / "predict.py"), "--demo", "--device", "cpu"],
    cwd=release_dir,
    env=offline_environment,
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=600,
)

if process.returncode != 0:
    raise RuntimeError(process.stderr)

demo = json.loads(process.stdout)
assert len(demo) == 4

for result in demo:
    for model_name in ["tfidf", "parsbert"]:
        for target in ["recommendation_status", "rating_band"]:
            probabilities = result[model_name][target]["probabilities"]
            assert abs(sum(probabilities.values()) - 1) < 1e-5
            assert all(0 <= probability <= 1 for probability in probabilities.values())

    print(f"\nText: {result['text']}")
    for model_name in ["tfidf", "parsbert"]:
        outputs = result[model_name]
        rec = outputs["recommendation_status"]
        band = outputs["rating_band"]
        print(
            f"{model_name}: {rec['label']} "
            f"({rec['probabilities'][rec['label']]:.1%})"
            f" | {band['label']} "
            f"({band['probabilities'][band['label']]:.1%})"
        )
    print(f"Tokens used: {result['token_count_used']} | Latency: {result['latency_ms']} ms")

(reports_dir / "offline_demo.json").write_text(
    json.dumps(demo, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("\nOffline CPU demo passed. Saved: offline_demo.json")

Running four offline prediction examples on CPU...

Text: خوبه
tfidf: recommended (98.3%) | high (51.7%)
parsbert: recommended (73.5%) | high (57.5%)
Tokens used: 3 | Latency: 455.33 ms

Text: معمولی بود، هنوز مطمئن نیستم ارزش خرید داشته باشه.
tfidf: no_idea (67.3%) | medium (83.0%)
parsbert: no_idea (94.6%) | medium (91.5%)
Tokens used: 13 | Latency: 123.97 ms

Text: بعد از دو روز خراب شد و اصلاً پیشنهاد نمی‌کنم.
tfidf: not_recommended (99.9%) | low (91.4%)
parsbert: not_recommended (99.2%) | low (95.2%)
Tokens used: 13 | Latency: 120.69 ms

Text: کیفیت ساخت خوبه ولی باتری زود تموم میشه و قیمتش بالاست.
tfidf: recommended (61.5%) | medium (58.1%)
parsbert: no_idea (74.9%) | medium (67.2%)
Tokens used: 14 | Latency: 135.56 ms

Offline CPU demo passed. Saved: offline_demo.json


In [10]:
# 1. Collect the project files and verify the source artifacts.

import hashlib
import json
import shutil
from pathlib import Path

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

root = Path("/content/drive/MyDrive/digikala-project-3")
release_dir = root / "release"
reports_dir = root / "reports"
splits_dir = Path("/content/digikala_splits")
release_dir.mkdir(exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256_file(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

test_files = {
    "TF-IDF": "baseline_test.json",
    "ParsBERT 1.2M": "parsbert_supervised_1200k_test.json",
    "Semi-supervised 1.2M": "parsbert_semi_supervised_1200k_test.json",
    "ParsBERT 3M": "parsbert_supervised_3000k_test.json",
}
test_results = {
    name: read_json(reports_dir / filename)
    for name, filename in test_files.items()
}
assert {result["total_rows"] for result in test_results.values()} == {919_675}

for target in ["recommendation_status", "rating_band"]:
    counts = {
        result["heads"][target]["evaluated_rows"]
        for result in test_results.values()
    }
    assert len(counts) == 1

base_errors = pd.read_csv(
    reports_dir / "error_cases_for_review.csv",
    dtype={"id": str},
    keep_default_na=False,
)
extra_errors = pd.read_csv(
    reports_dir / "sarcasm_candidates.csv",
    dtype={"id": str},
    keep_default_na=False,
)
all_errors = pd.concat([base_errors, extra_errors], ignore_index=True)
assert all_errors["id"].is_unique
assert len(all_errors) == 32
assert all_errors["manually_reviewed"].astype(str).str.lower().eq("true").all()

demo = read_json(reports_dir / "offline_demo.json")
assert len(demo) == 4

# Record inferred CSV types and the exact source-file fingerprint.
print("Recording dataset schema...", flush=True)
raw_path = Path(hf_hub_download(
    repo_id="RadeAI/Digikala_comments_products",
    filename="digikala-comments.csv",
    repo_type="dataset",
))
with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    db.execute(
        "SELECT * FROM read_csv(?, header=true, sample_size=-1) LIMIT 0",
        [str(raw_path)],
    )
    schema = pd.DataFrame([
        {"column": field[0], "inferred_type": str(field[1])}
        for field in db.description
    ])

assert len(schema) == 15
schema.to_csv(reports_dir / "dataset_schema.csv", index=False)

# Measure overlap for the actual selected 3M training subset.
print("Measuring exact-text overlap for the selected 3M subset...", flush=True)
with duckdb.connect(config={"memory_limit": "3GB", "threads": "2"}) as db:
    overlap_row = db.execute("""
        WITH selected AS MATERIALIZED (
            SELECT id, body
            FROM read_csv(?, header=true, all_varchar=true)
            ORDER BY substr(sha256('2026|' || id), 1, 16), id
            LIMIT 3000000
        ), train_texts AS MATERIALIZED (
            SELECT DISTINCT body FROM selected
        )
        SELECT count(*) AS test_rows,
               count(t.body) AS test_rows_with_text_in_selected_train
        FROM read_csv(?, header=true, all_varchar=true) e
        LEFT JOIN train_texts t USING (body)
    """, [
        str(splits_dir / "train_supervised.csv"),
        str(splits_dir / "test.csv"),
    ]).fetchone()

selected_overlap = {
    "training_rows": 3_000_000,
    "test_rows": int(overlap_row[0]),
    "test_rows_with_text_in_selected_train": int(overlap_row[1]),
    "test_overlap_pct": 100 * overlap_row[1] / overlap_row[0],
}
(reports_dir / "selected_3m_text_overlap.json").write_text(
    json.dumps(selected_overlap, indent=2), encoding="utf-8",
)

manifest = read_json(splits_dir / "split_manifest.json")
manifest["source_file"] = {
    "repository": "RadeAI/Digikala_comments_products",
    "filename": raw_path.name,
    "bytes": raw_path.stat().st_size,
    "sha256": sha256_file(raw_path),
}
manifest["cleaning_policy"] = {
    "missing_text_markers": ["", "nan"],
    "multiple_texts_for_same_id": "Exclude the complete ID group",
    "same_text_for_same_id": "Merge to one row",
    "conflicting_valid_recommendations": "Mask recommendation target",
    "conflicting_valid_rating_bands": "Mask rating-band target",
    "invalid_numeric_rating": "Mask rating-band target",
    "training_rows_without_either_target": "Exclude from train_supervised",
}
manifest["selected_3m_text_overlap"] = selected_overlap

data_dir = release_dir / "data"
data_dir.mkdir(exist_ok=True)
for filename in [
    "train_supervised.csv",
    "train_pseudo_unlabeled.csv",
    "natural_unlabeled.csv",
    "validation.csv",
    "test.csv",
]:
    print(f"Copying {filename}...", flush=True)
    shutil.copyfile(splits_dir / filename, data_dir / filename)

(data_dir / "split_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8",
)

# Preserve the files used by the Colab notebooks.
print("Collecting experiment artifacts...", flush=True)
for folder in [
    "parsbert_supervised",
    "parsbert_supervised_1200k",
    "parsbert_supervised_3000k",
    "parsbert_semi_supervised_1200k",
]:
    shutil.copytree(
        root / "artifacts" / folder,
        release_dir / "artifacts" / folder,
        dirs_exist_ok=True,
    )

for filename in [
    "tfidf_vectorizer.joblib",
    "tfidf_recommendation_status.joblib",
    "tfidf_rating_band.joblib",
]:
    shutil.copyfile(
        root / "artifacts" / filename,
        release_dir / "artifacts" / filename,
    )

for filename in [
    "digikala_transformer.py",
    "digikala_training.py",
    "digikala_evaluation.py",
    "normalize_persian.py",
]:
    shutil.copyfile(
        root / "artifacts" / "parsbert_supervised" / filename,
        release_dir / filename,
    )

for filename in ["digikala_inputs.tar.gz", "digikala_project.duckdb"]:
    shutil.copyfile(root / filename, release_dir / filename)

error_report_path = reports_dir / "error_analysis.md"
error_report = error_report_path.read_text(encoding="utf-8").replace(
    "> # تحلیل ۲۴ خطای مدل منتخب",
    "> # تحلیل ۳۲ خطای مدل منتخب",
    1,
)
error_report_path.write_text(error_report, encoding="utf-8")
shutil.copytree(reports_dir, release_dir / "reports", dirs_exist_ok=True)

# Find the original notebook files without changing their contents.
notebook_names = [
    "01_descriptive_analysis.ipynb",
    "02_baselines.ipynb",
    "03_transfer_learning_finetuning.ipynb",
    "04_semi_supervised.ipynb",
]
search_dirs = [
    root / "notebooks",
    Path("/content"),
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/drive/MyDrive"),
]
notebooks_dir = release_dir / "notebooks"
notebooks_dir.mkdir(exist_ok=True)
notebook_inventory = {}

for name in notebook_names:
    matches = []
    for folder in search_dirs:
        if folder.is_dir():
            matches.extend(
                path for path in folder.iterdir()
                if path.is_file()
                and path.name.removesuffix(".ipynb") == name.removesuffix(".ipynb")
            )

    if matches:
        source = max(matches, key=lambda path: path.stat().st_mtime)
        notebook = read_json(source)
        assert notebook["nbformat"] == 4 and isinstance(notebook["cells"], list)
        shutil.copyfile(source, notebooks_dir / name)
        notebook_inventory[name] = {
            "source": str(source),
            "cells": len(notebook["cells"]),
        }

missing_notebooks = [
    name for name in notebook_names if name not in notebook_inventory
]
print(f"\nNotebook files found: {len(notebook_inventory)} / 4")
for name in notebook_inventory:
    print(f"FOUND: {name}")
for name in missing_notebooks:
    print(f"MISSING: {name}")
print(f"Reviewed error cases: {len(all_errors)}")
print(f"Selected-training exact-text overlap: {selected_overlap['test_overlap_pct']:.2f}%")
print("Project files collected.")

Recording dataset schema...
Measuring exact-text overlap for the selected 3M subset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Copying train_supervised.csv...
Copying train_pseudo_unlabeled.csv...
Copying natural_unlabeled.csv...
Copying validation.csv...
Copying test.csv...

Notebook files found: 4 / 4
FOUND: 01_descriptive_analysis.ipynb
FOUND: 02_baselines.ipynb
FOUND: 03_transfer_learning_finetuning.ipynb
FOUND: 04_semi_supervised.ipynb
Reviewed error cases: 32
Selected-training exact-text overlap: 33.71%
Project files collected.


In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive, files
from IPython.display import Image, display

drive.mount("/content/drive")

uploaded = files.upload()
filename = "finalize_digikala_project.py"

if filename not in uploaded:
    raise ValueError(f"Select {filename}")

script_path = Path("/content") / filename
script_path.write_bytes(uploaded[filename])

result = subprocess.run([
    sys.executable, "-u", str(script_path),
])

if result.returncode != 0:
    print("FINALIZATION_INCOMPLETE — send the last output.")
else:
    figure = Path(
        "/content/drive/MyDrive/digikala-project-3/"
        "release/reports/figures/test_macro_f1.png"
    )
    if figure.is_file():
        display(Image(filename=str(figure)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving finalize_digikala_project.py to finalize_digikala_project.py
